Github repo link: https://github.com/Timthn/UST_COMP_Course_Workload_Prediction

# Part1: manual adding the cookie and get the data for the course you choose
# iif selenium fails, try this part 

In [5]:
!pip install browser-cookie3 requests

In [3]:
import requests
import json
import pandas as pd
import os

In [1]:


def fetch_and_save_reviews(course_name, cookie_str, output_root="./ust_reviews"):
    # 1. Setup URL and Parameters( the request library make the url look like this https://ust.space/review/COMP1021/get?single=false&composer=false&preferences%5Bsort%5D=0&preferences%5BfilterInstructor%5D=0&preferences%5BfilterSemester%5D=0&preferences%5BfilterRating%5D=0) )
    url = f"https://ust.space/review/{course_name}/get"
    params = {
        "single": "false",
        "composer": "false",
        "preferences[sort]": 0,
        "preferences[filterInstructor]": 0,
        "preferences[filterSemester]": 0,
        "preferences[filterRating]": 0
    }
    
    # 2. Setup Headers (Simulating browser behavior)( simulate the human browser header, especially the cookie and user agent)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Cookie": cookie_str
    }

    # 3. Send Request
    print(f"Fetching data for {course_name}...")
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for HTTP errors
        data = response.json()
        reviews = data.get('reviews', [])
        print(f"Successfully retrieved {len(reviews)} reviews!")
    except Exception as e:
        print(f"Failed to fetch data: {e}")
        return

    if not reviews:
        print("No review data found. Process stopped.")
        return

    # 4. Prepare Output Directory: ust_reviews/<course_name>
    output_folder = os.path.join(output_root, course_name)
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder: {output_folder}")

    json_path = os.path.join(output_folder, f"{course_name}_reviews.json")
    csv_path = os.path.join(output_folder, f"{course_name}_reviews.csv")

    # 5. Save as JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(reviews, f, ensure_ascii=False, indent=4)
    print(f"JSON saved to: {json_path}")

    # 6. Save as CSV (encoding good for chinese comment and excel )
    df = pd.DataFrame(reviews)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"CSV saved to: {csv_path}")



In [ ]:
# --- Main Entry Point ---
if __name__ == "__main__":
    # Configuration
    # Set env var USTSPACE_COOKIE in terminal before running.

    MY_COOKIE = os.getenv("USTSPACE_COOKIE", "")
    if not MY_COOKIE:
        raise ValueError("Missing env var: USTSPACE_COOKIE")

    TARGET_COURSES = ["MATH1013"]  # change target courses here
    SAVE_ROOT = os.getenv("SAVE_ROOT", "./ust_reviews")

    for course in TARGET_COURSES:
        fetch_and_save_reviews(course, MY_COOKIE, SAVE_ROOT)

Fetching data for MATH1013...
Successfully retrieved 1079 reviews!
Output folder: ./ust_reviews\MATH1013
JSON saved to: ./ust_reviews\MATH1013\MATH1013_reviews.json
CSV saved to: ./ust_reviews\MATH1013\MATH1013_reviews.csv


# Part2: using selenium to get the cookie and run the script for testing 

In [ ]:
# package installation for selenium, which is used for the authentication part (if you want to use selenium to get the cookie, otherwise you can just copy the cookie from the browser and paste it in the code above)
pip install selenium


  Obtaining dependency information for selenium from https://files.pythonhosted.org/packages/a8/d6/e4160989ef6b272779af6f3e5c43c3ba9be6687bdc21c68c3fb220e555b3/selenium-4.41.0-py3-none-any.whl.metadata
  Using cached selenium-4.41.0-py3-none-any.whl.metadata (7.5 kB)
  Obtaining dependency information for certifi>=2026.1.4 from https://files.pythonhosted.org/packages/9a/3c/c17fb3ca2d9c3acff52e30b309f538586f9f5b9c9cf454f3845fc9af4881/certifi-2026.2.25-py3-none-any.whl.metadata
  Obtaining dependency information for trio<1.0,>=0.31.0 from https://files.pythonhosted.org/packages/1c/93/dab25dc87ac48da0fe0f6419e07d0bfd98799bed4e05e7b9e0f85a1a4b4b/trio-0.33.0-py3-none-any.whl.metadata
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Obtaining dependency information for trio-websocket<1.0,>=0.12.2 from https://files.pythonhosted.org/packages/c7/19/eb640a397bba49ba49ef9dbe2e7e5c04202ba045b6ce2ec36e9cadc51e04/trio_websocket-0.12.2-py3-none-any.whl.metadata
  Using cached trio_we

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 2.1.1 requires sentencepiece, which is not installed.
botocore 1.27.59 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
httpcore 1.0.5 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.


In [20]:
pip install --upgrade typing_extensions

In [1]:
import os
import time
import json
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


In [2]:
# ==========================================
# 1. Automated Login & Cookie Extraction Module
# ==========================================
def _click_first_available(driver, wait, locators, step_name):
    """Try multiple locators and click the first matched clickable element."""
    for by, value in locators:
        try:
            elem = wait.until(EC.element_to_be_clickable((by, value)))
            driver.execute_script("arguments[0].click();", elem)
            print(f"[OK] {step_name}")
            return True
        except Exception:
            continue
    print(f"[WARN] {step_name} not found, continue...")
    return False


def _switch_to_latest_window(driver):
    """If login opens a new tab/window, switch to it."""
    handles = driver.window_handles
    if len(handles) > 1:
        driver.switch_to.window(handles[-1])


def _find_element_any_frame(driver, wait, locator):
    """Find an element in default content or any iframe."""
    driver.switch_to.default_content()
    try:
        return wait.until(EC.presence_of_element_located(locator))
    except Exception:
        pass

    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    for frame in iframes:
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(frame)
            return wait.until(EC.presence_of_element_located(locator))
        except Exception:
            continue

    driver.switch_to.default_content()
    raise Exception(f"Element not found in default content or iframes: {locator}")


def _type_first_available(driver, wait, locators, value, field_name):
    """Type value into the first matched input field."""
    for locator in locators:
        try:
            elem = _find_element_any_frame(driver, wait, locator)
            elem.clear()
            elem.send_keys(value)
            print(f"[OK] Filled {field_name}")
            return True
        except Exception:
            continue
    raise Exception(f"Cannot find {field_name} input")


def get_auth_cookie(username, password, login_url="https://ust.space"):
    print("Launching browser for guided login flow...")

    options = webdriver.ChromeOptions()
    options.add_argument('--log-level=3')
    # Keep browser visible for login steps.

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 25)
    cookie_str = ""

    try:
        driver.get(login_url)

        # Step 1: click "Join Us Now"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(., 'Join Us Now')]|//button[contains(., 'Join Us Now')]|//*[contains(., 'Join Us Now') and (self::a or self::button)]"),
                (By.XPATH, "//a[contains(., 'Join Us')]|//button[contains(., 'Join Us')]")
            ],
            "Clicked Join Us Now"
        )

        # Step 2: click "Return to login page"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(., 'Return to login page')]|//button[contains(., 'Return to login page')]|//*[contains(., 'Return to login page') and (self::a or self::button)]"),
                (By.XPATH, "//a[contains(., 'Return to log in')]|//button[contains(., 'Return to log in')]"),
                (By.XPATH, "//a[contains(., 'Log in')]|//button[contains(., 'Log in')]|//a[contains(., 'Sign in')]|//button[contains(., 'Sign in')]")
            ],
            "Clicked Return to login page"
        )

        # Login may open in a new tab/window
        time.sleep(1)
        _switch_to_latest_window(driver)

        # Step 3: fill USTSPACE login form
        _type_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//input[contains(@placeholder, 'ITSC Network Account Name')]"),
                (By.XPATH, "//input[contains(@name, 'username') or contains(@id, 'username') or contains(@name, 'account')]"),
                (By.XPATH, "//input[@type='text']")
            ],
            username,
            "username"
        )

        _type_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//input[contains(@placeholder, 'Password')]"),
                (By.XPATH, "//input[@type='password']"),
                (By.XPATH, "//input[contains(@name, 'password') or contains(@id, 'password')]")
            ],
            password,
            "password"
        )

        driver.switch_to.default_content()

        # Submit login form
        clicked_login = _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//button[contains(., 'Login')]|//a[contains(., 'Login')]"),
                (By.XPATH, "//input[@type='submit']")
            ],
            "Clicked Login"
        )
        if not clicked_login:
            # Fallback: press Enter in password field
            try:
                pass_input = _find_element_any_frame(driver, wait, (By.XPATH, "//input[@type='password']"))
                pass_input.send_keys(Keys.RETURN)
                print("[OK] Submitted form with ENTER")
            except Exception:
                pass

        # Return focus to latest window after auth redirects
        time.sleep(1)
        _switch_to_latest_window(driver)

        # Step 4: click "Course Review"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'course review')]|//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'course review')]"),
                (By.XPATH, "//a[contains(@href, '/review')]|//button[contains(@onclick, 'review')]")
            ],
            "Clicked Course Review"
        )

        # Step 5: get cookies after navigation to review area
        print("Waiting for authentication tokens...")
        for _ in range(30):
            cookies = {c['name']: c['value'] for c in driver.get_cookies()}
            if 'ustspace_session' in cookies and 'XSRF-TOKEN' in cookies:
                cookie_str = "; ".join([f"{k}={v}" for k, v in cookies.items()])
                print("-> Authentication successful. Cookie acquired.")
                break
            time.sleep(1)

    except Exception as e:
        print(f"Failed to acquire cookie: {e}")

    finally:
        driver.quit()

    return cookie_str

# --- Example Usage ---
# my_cookie = get_auth_cookie("your_account", "your_password")
# print(my_cookie)

In [3]:

# ==========================================
# 2. Scraper Module
# ==========================================
def fetch_and_save_reviews(course_name, cookie_str, output_root="./ust_reviews"):
    url = f"https://ust.space/review/{course_name}/get"
    params = {
        "single": "false", "composer": "false",
        "preferences[sort]": 0, "preferences[filterInstructor]": 0,
        "preferences[filterSemester]": 0, "preferences[filterRating]": 0
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Cookie": cookie_str
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() 
        reviews = response.json().get('reviews', [])
        
        if not reviews:
            print(f"[{course_name}] No reviews found. Skipping.")
            return

        # Ensure directory exists: ust_reviews/<course_name>
        output_folder = os.path.join(output_root, course_name)
        os.makedirs(output_folder, exist_ok=True)
        print(f"Output folder: {output_folder}")

        json_path = os.path.join(output_folder, f"{course_name}_reviews.json")
        csv_path = os.path.join(output_folder, f"{course_name}_reviews.csv")

        # Save JSON & CSV
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(reviews, f, ensure_ascii=False, indent=4)
        
        pd.DataFrame(reviews).to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"[{course_name}] Saved {len(reviews)} reviews.")

    except Exception as e:
        print(f"[{course_name}] Fetch failed: {e}")


Testing the web scrabing with course list with two items

In [ ]:
# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == "__main__":
    # Credentials are loaded from environment variables.
    MY_ACCOUNT = os.getenv("USTSPACE_ACCOUNT", "")
    MY_PASSWORD = os.getenv("USTSPACE_PASSWORD", "")

    if not MY_ACCOUNT or not MY_PASSWORD:
        raise ValueError("Missing env vars: USTSPACE_ACCOUNT / USTSPACE_PASSWORD")

    SAVE_ROOT = os.getenv("SAVE_ROOT", "./ust_reviews")
    TARGET_COURSES = ["COMP1021", "COMP2011"]

    # 1. Authenticate
    auth_cookie = get_auth_cookie(MY_ACCOUNT, MY_PASSWORD)

    # 2. Scrape Data
    if auth_cookie:
        print("\nStarting data extraction...")
        for course in TARGET_COURSES:
            fetch_and_save_reviews(course, auth_cookie, SAVE_ROOT)
            time.sleep(2)  # Polite delay between server requests

        print("\nAll tasks completed successfully!")
    else:
        print("\nProgram terminated due to missing authentication cookie.")

Launching browser for guided login flow...
[OK] Clicked Join Us Now
[OK] Clicked Return to login page
[OK] Filled username
[OK] Filled password
[OK] Clicked Login
[OK] Clicked Course Review
Waiting for authentication tokens...
-> Authentication successful. Cookie acquired.

Starting data extraction...
Output folder: ./ust_reviews\COMP1021
[COMP1021] Saved 1234 reviews.
Output folder: ./ust_reviews\COMP2011
[COMP2011] Saved 311 reviews.

All tasks completed successfully!


# Part3: Our task is to web scrab the comp courses with reviews. We get the links from ustspace with the same cookie and get the list, clean the courses with review and  sort with decending reviews numbers.

In [ ]:
import re
import requests

def get_comp_course_list(year="2025-26"):
    url = f"https://prog-crs.hkust.edu.hk/ugcourse/{year}/COMP/"
    html = requests.get(url, timeout=20).text

    # get comp+ 4 digit + optional letter (eg COMP3511H)
    courses = re.findall(r"\bCOMP\s*\d{4}[A-Z]?\b", html, flags=re.IGNORECASE)

    # cleaning and sorting with reviews
    course_list = sorted({c.replace(" ", "").upper() for c in courses})
    return course_list

In [7]:
comp_list = get_comp_course_list("2025-26")
print(comp_list)
print(len(comp_list))

['COMP1001', 'COMP1021', 'COMP1022P', 'COMP1022Q', 'COMP1023', 'COMP1028', 'COMP1029C', 'COMP1029J', 'COMP1029P', 'COMP1029V', 'COMP1942', 'COMP1943', 'COMP1944', 'COMP1945', 'COMP1991', 'COMP2011', 'COMP2012', 'COMP2012H', 'COMP2211', 'COMP2611', 'COMP2633', 'COMP2711', 'COMP2711H', 'COMP3021', 'COMP3031', 'COMP3071', 'COMP3111', 'COMP3111H', 'COMP3211', 'COMP3311', 'COMP3511', 'COMP3631', 'COMP3632', 'COMP3633', 'COMP3711', 'COMP3711H', 'COMP3721', 'COMP4021', 'COMP4121', 'COMP4211', 'COMP4221', 'COMP4222', 'COMP4321', 'COMP4331', 'COMP4332', 'COMP4411', 'COMP4421', 'COMP4431', 'COMP4441', 'COMP4451', 'COMP4461', 'COMP4462', 'COMP4471', 'COMP4521', 'COMP4531', 'COMP4541', 'COMP4551', 'COMP4611', 'COMP4621', 'COMP4631', 'COMP4632', 'COMP4633', 'COMP4634', 'COMP4635', 'COMP4641', 'COMP4651', 'COMP4900', 'COMP4901', 'COMP4901K', 'COMP4901M', 'COMP4901N', 'COMP4901O', 'COMP4901P', 'COMP4901S', 'COMP4901U', 'COMP4901W', 'COMP4901Y', 'COMP4910', 'COMP4911', 'COMP4971', 'COMP4981', 'COMP498

In [ ]:
# sort the list of courses into those with reviews and those without reviews, and also count the number of reviews for each course, and also count the number of failed requests (if any)

def check_comp_review_status(course_list, cookie_str, sleep_sec=0.2):
    params = {
        "single": "false",
        "composer": "false",
        "preferences[sort]": 0,
        "preferences[filterInstructor]": 0,
        "preferences[filterSemester]": 0,
        "preferences[filterRating]": 0
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Cookie": cookie_str
    }

    has_reviews = []   # [(course_code, review_count), ...]
    no_reviews = []    # [course_code, ...]
    failed = []        # [(course_code, error_msg), ...]

    s = requests.Session()

    for course in course_list:
        url = f"https://ust.space/review/{course}/get"
        try:
            r = s.get(url, headers=headers, params=params, timeout=15)
            r.raise_for_status()
            reviews = r.json().get("reviews", [])
            n = len(reviews)

            if n > 0:
                has_reviews.append((course, n))
            else:
                no_reviews.append(course)

        except Exception as e:
            failed.append((course, str(e)))

        time.sleep(sleep_sec)

    # sort courses with reviews by review count descending
    has_reviews.sort(key=lambda x: x[1], reverse=True)
    return has_reviews, no_reviews, failed

In [ ]:
# Here is the main loop of sorting courses based on review status
#  comp_list has been generated in above cell
# comp_list = get_comp_course_list("2025-26")

has_reviews, no_reviews, failed = check_comp_review_status(comp_list, auth_cookie)

print("has review:", len(has_reviews))
print("no review:", len(no_reviews))
print("failed:", len(failed))

# course List for main loop
comp_with_reviews = [c for c, _ in has_reviews]
comp_without_reviews = no_reviews

print("has review:", comp_with_reviews[:20])
print("no review:", comp_without_reviews[:20])

has review: 78
no review: 10
failed: 0
has review: ['COMP1021', 'COMP1022P', 'COMP2011', 'COMP1022Q', 'COMP2711', 'COMP2012', 'COMP2012H', 'COMP1001', 'COMP1942', 'COMP3711', 'COMP3511', 'COMP2711H', 'COMP1023', 'COMP2611', 'COMP3111', 'COMP2211', 'COMP3021', 'COMP3311', 'COMP5212', 'COMP1029C']
no review: ['COMP1945', 'COMP3071', 'COMP4551', 'COMP4901', 'COMP4901M', 'COMP4901P', 'COMP4901S', 'COMP4911', 'COMP4971', 'COMP4981H']


In [20]:
# check the two lists in a df form

max_len = max(len(comp_with_reviews), len(comp_without_reviews))

with_reviews_col = comp_with_reviews + [""] * (max_len - len(comp_with_reviews))
without_reviews_col = comp_without_reviews + [""] * (max_len - len(comp_without_reviews))

df_courses = pd.DataFrame({
    "courses_with_reviews": with_reviews_col,
    "courses_without_reviews": without_reviews_col
})

display(df_courses)

,courses_with_reviews,courses_without_reviews
0,COMP1021,COMP1945
1,COMP1022P,COMP3071
2,COMP2011,COMP4551
3,COMP1022Q,COMP4901
4,COMP2711,COMP4901M
...,...,...
73,COMP4901K,
74,COMP4901O,
75,COMP1991,
76,COMP4910,


In [21]:
df_courses.to_csv("./ust_reviews/comp_review_status_table.csv", index=False, encoding="utf-8-sig")

In [ ]:
## final target, call the main loop with the above course lists to fetch and save the reviews for courses with reviews, and also print out the courses without reviews for reference

# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == "__main__":
    # Credentials are loaded from environment variables.
    MY_ACCOUNT = os.getenv("USTSPACE_ACCOUNT", "")
    MY_PASSWORD = os.getenv("USTSPACE_PASSWORD", "")

    if not MY_ACCOUNT or not MY_PASSWORD:
        raise ValueError("Missing env vars: USTSPACE_ACCOUNT / USTSPACE_PASSWORD")

    SAVE_ROOT = os.getenv("SAVE_ROOT", "./ust_reviews")
    TARGET_COURSES = comp_with_reviews  # only fetch reviews for courses that have reviews, and skip those without reviews

    # 1. Authenticate
    auth_cookie = get_auth_cookie(MY_ACCOUNT, MY_PASSWORD)

    # 2. Scrape Data
    if auth_cookie:
        print("\nStarting data extraction...")
        for course in TARGET_COURSES:
            fetch_and_save_reviews(course, auth_cookie, SAVE_ROOT)
            time.sleep(2)  # Polite delay between server requests

        print("\nAll tasks completed successfully!")
    else:
        print("\nProgram terminated due to missing authentication cookie.")

Launching browser for guided login flow...
[OK] Clicked Join Us Now
[OK] Clicked Return to login page
[OK] Filled username
[OK] Filled password
[OK] Clicked Login
[OK] Clicked Course Review
Waiting for authentication tokens...
-> Authentication successful. Cookie acquired.

Starting data extraction...
Output folder: ./ust_reviews\COMP1021
[COMP1021] Saved 1234 reviews.
Output folder: ./ust_reviews\COMP1022P
[COMP1022P] Saved 356 reviews.
Output folder: ./ust_reviews\COMP2011
[COMP2011] Saved 311 reviews.
Output folder: ./ust_reviews\COMP1022Q
[COMP1022Q] Saved 231 reviews.
Output folder: ./ust_reviews\COMP2711
[COMP2711] Saved 177 reviews.
Output folder: ./ust_reviews\COMP2012
[COMP2012] Saved 109 reviews.
Output folder: ./ust_reviews\COMP2012H
[COMP2012H] Saved 75 reviews.
Output folder: ./ust_reviews\COMP1001
[COMP1001] Saved 74 reviews.
Output folder: ./ust_reviews\COMP1942
[COMP1942] Saved 73 reviews.
Output folder: ./ust_reviews\COMP3711
[COMP3711] Saved 73 reviews.
Output folder:

# Part 4： After web scrabing, we wish to visualized the data count we scrabbed and analysize on the data sample  

In [24]:
from pathlib import Path
import re

root = Path("./ust_reviews")
rows = []

# scan path like ust_reviews/COMP1021/COMP1021_reviews.csv 
for csv_file in sorted(root.glob("COMP*/COMP*_reviews.csv")):
    course = csv_file.stem.replace("_reviews", "")  # e.g. COMP1021
    # save the course name, review count, and output path to a list of dicts, but only for valid course names (COMP + 4 digit + optional letter)
    if not re.match(r"^COMP\d{4}[A-Z]?$", course):
        continue

    try:
        review_count = len(pd.read_csv(csv_file))
    except Exception:
        review_count = 0

    output_folder = str(csv_file.parent)  

    rows.append({
        "course name": course,
        "course review number": review_count,
        "Output path": output_folder
    })

# save as table view   
df_summary = pd.DataFrame(rows).sort_values(
    by=["course review number", "course name"],
    ascending=[False, True]
).reset_index(drop=True)

# save path 
out_path = root / "comp_course_review_summary.csv"
df_summary.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Saved: {out_path}")
display(df_summary.head(20))

Saved: ust_reviews\comp_course_review_summary.csv


,course name,course review number,Output path
0,COMP1021,1234,ust_reviews\COMP1021
1,COMP1022P,356,ust_reviews\COMP1022P
2,COMP2011,311,ust_reviews\COMP2011
3,COMP1022Q,231,ust_reviews\COMP1022Q
4,COMP2711,177,ust_reviews\COMP2711
5,COMP2012,109,ust_reviews\COMP2012
6,COMP2012H,75,ust_reviews\COMP2012H
7,COMP1001,74,ust_reviews\COMP1001
8,COMP1942,73,ust_reviews\COMP1942
9,COMP3711,73,ust_reviews\COMP3711


In [25]:
df_summary["course review number"].sum()
print(f"Total reviews across all COMP courses: {df_summary['course review number'].sum()}")

Total reviews across all COMP courses: 3904


In [26]:

df_summary["course name"] = df_summary["course name"].astype(str)
df_summary["course review number"] = pd.to_numeric(df_summary["course review number"], errors="coerce").fillna(0)

# split the dataframe into 4 dataframes based on the level of the course (1000-level, 2000-level, 3000-level, 4000-level)
df_level_1 = df_summary[df_summary["course name"].str[4] == "1"].copy()
df_level_2 = df_summary[df_summary["course name"].str[4] == "2"].copy()
df_level_3 = df_summary[df_summary["course name"].str[4] == "3"].copy()
df_level_4 = df_summary[df_summary["course name"].str[4] == "4"].copy()
df_level_5 = df_summary[df_summary["course name"].str[4] == "5"].copy()
# sum the review numbers for each level
sum_reviews_1 = int(df_level_1["course review number"].sum())
sum_reviews_2 = int(df_level_2["course review number"].sum())
sum_reviews_3 = int(df_level_3["course review number"].sum())
sum_reviews_4 = int(df_level_4["course review number"].sum())
sum_reviews_5 = int(df_level_5["course review number"].sum())

print("1000-level total reviews:", sum_reviews_1)
print("2000-level total reviews:", sum_reviews_2)
print("3000-level total reviews:", sum_reviews_3)
print("4000-level total reviews:", sum_reviews_4)
print("5000-level total reviews:", sum_reviews_5)

1000-level total reviews: 2189
2000-level total reviews: 871
3000-level total reviews: 424
4000-level total reviews: 316
5000-level total reviews: 104


In [28]:
count_courses_1 = len(df_level_1)
count_courses_2 = len(df_level_2)
count_courses_3 = len(df_level_3)
count_courses_4 = len(df_level_4)
count_courses_5 = len(df_level_5)

print("1000-level course with comment count:", count_courses_1)
print("2000-level course with comment count:", count_courses_2)
print("3000-level course with comment count:", count_courses_3)
print("4000-level course with comment count:", count_courses_4)
print("5000-level course with comment count:", count_courses_5)

1000-level course with comment count: 14
2000-level course with comment count: 8
3000-level course with comment count: 13
4000-level course with comment count: 37
5000-level course with comment count: 6


In [13]:
# check data quality:
df_sample_review=pd.read_csv(root / "COMP1021/COMP1021_reviews.csv")
display(df_sample_review.head(5))
print(f"feature number= column number: {df_sample_review.shape[1]}")
feature_names = df_sample_review.columns.tolist()
print("feature names:", feature_names)  

,hash,semester,instructors,is_author,author,date,title,comment_content,comment_teaching,comment_grading,...,has_project,has_attendance,has_reading,has_presentation,upvote_count,vote_count,voted,is_upvote,comment_count,attachments
0,M3oDtgrPF6OeqNhnlsjSISZITTv5sBnX,2016-17 Fall,"[{'id': 99, 'name': 'LAM, Gibson', 'rating': 1}]",False,Harry Ho,"Jan 21, 2017",Assignment 易，Mid-term 淺，Final 難,梗要：<br />* 以 Python 3 + PyGame 作藍本，教授 programm...,"L1 &amp; L2: Gibson LAM, gibson@cse.ust.hk<br ...","佔分比例：<br />【1】Mid-term Exam (open-book, 1 hr) ...",...,False,False,False,False,160,164,False,False,1,[]
1,tphFzlcv6Vjt46bXrFBt5m48m7qkj05Q,2024-25 Fall,"[{'id': 99, 'name': 'LAM, Gibson', 'rating': 1}]",False,ustreviews,"Dec 10, 2024","A terrible introduction to computer science, a...",When one looks to take an introduction to comp...,I can only speak to Gibson Lam&#039;s teaching...,An area this course redeems itself a little is...,...,True,False,False,False,20,20,False,False,4,[]
2,t2wiB4FkgpCBj48l8lxEKy3rXpzps04s,2019-20 Spring,"[{'id': 95, 'name': 'ROSSITER, David', 'rating...",False,garbage,"May 26, 2020",Why are all the exams this semester so hellish...,Basic stuff of Python. This semester excludes ...,David is a really funny and nice guy. The TA n...,The midterm exam is really simple. The mean is...,...,False,False,False,False,21,22,False,False,0,[]
3,E3BqP0iPHegwW5XULLWiMqQIBz1tdky6,2021-22 Fall,"[{'id': 799, 'name': 'LAM, Ngok', 'rating': 1}...",False,Saul Goodman,"Aug 18, 2022",Saul Goodman&#039;s honest review,Russian guy here after finishing Better Call S...,Prof David is definitely a good teacher. He is...,There are 2 marking methods in this course jus...,...,False,False,False,False,18,19,False,False,1,"[{'key': 'aYVqbAqedA3gH0BXdxztll89L2KTx4bW', '..."
4,A38ZjP32gxXPwInBWOfuw2Wakbk5n2to,2021-22 Fall,"[{'id': 799, 'name': 'LAM, Ngok', 'rating': 1}]",False,mark_,"Dec 09, 2021",human aptitude / geometry test disguised as pr...,"Basic Python stuff. If/else, for loops, lists,...",Alex Lam (my professor) is good at teaching an...,Two grading methods.<br /><br />24% midterm + ...,...,False,False,False,False,16,20,False,False,0,[]


feature number= column number: 30
feature names: ['hash', 'semester', 'instructors', 'is_author', 'author', 'date', 'title', 'comment_content', 'comment_teaching', 'comment_grading', 'comment_workload', 'rating_content', 'rating_teaching', 'rating_grading', 'rating_workload', 'has_midterm', 'has_final', 'has_quiz', 'has_assignment', 'has_essay', 'has_project', 'has_attendance', 'has_reading', 'has_presentation', 'upvote_count', 'vote_count', 'voted', 'is_upvote', 'comment_count', 'attachments']


# Part 5 : Get additional data from the website and its data repo in github

get the insturctor list from ust-ranking 

 https://ust-rankings.com/


data source= https://raw.githubusercontent.com/ust-archive/ust-rankings-data/data/ratings-instructor.json
 there is a ranking of insturctors from this website  e.g A+ A A- B+, B, B-, C+, C, C-, D, E, F  scab the source data and get the instructor list and their ranking save it as a dataframe with two columns: instructor name and ranking

 Based on the calculation Of the raw datathe website: formula  
content.bayesian * 2/3 * 0.4 + 
teaching.bayesian * 2/3 * 0.4 + 
grading.bayesian * 2/3 * 0.15 + 
workload.bayesian * 2/3 * 0.05 + 
course.bayesian * 1/3 * 0.25 + 
instructor.bayesian * 1/3 * 0.

we change it by removing the workload to prevent data leakage:

def calc_score(row, term):
    # Remove workload to avoid leakage into workload prediction target
    score_wo_workload = (
        get_value(row, "content", "bayesian", term) * 2/3 * 0.4 +
        get_value(row, "teaching", "bayesian", term) * 2/3 * 0.4 +
        get_value(row, "grading", "bayesian", term) * 2/3 * 0.15 +
        get_value(row, "course", "bayesian", term) * 1/3 * 0.25 +
        get_value(row, "instructor", "bayesian", term) * 1/3 * 0.75
    )

    # Old total weight was 1.0; after removing workload (2/3*0.05 = 1/30),
    # remaining weight is 29/30. Normalize back to comparable scale.
    return score_wo_workload / (29/30)


 the website rank the instructor based on the above formula, 
  then assign a letter grade based on the score: 
≥0.9 = A+

≥0.8 = A

≥0.75 = A-

≥0.6 = B+

≥0.45 = B

≥0.35 = B-

≥0.3 = C+

≥0.25 = C

≥0.2 = C-

≥0.1 = D

remaining = F


In [42]:
import requests
import pandas as pd

url = "https://raw.githubusercontent.com/ust-archive/ust-rankings-data/data/ratings-instructor.json"
data = requests.get(url).json()
df = pd.json_normalize(data)

print(df.columns.tolist())
display(df.head())

['meta.name', 'meta.courses.100', 'meta.courses.101', 'meta.courses.102', 'meta.courses.103', 'meta.courses.94', 'meta.courses.95', 'meta.courses.96', 'meta.courses.97', 'meta.courses.98', 'meta.courses.99', 'meta.courses.78', 'meta.courses.79', 'meta.courses.80', 'meta.courses.81', 'meta.courses.82', 'meta.courses.83', 'meta.courses.84', 'meta.courses.85', 'meta.courses.86', 'meta.courses.87', 'meta.courses.88', 'meta.courses.89', 'meta.courses.90', 'meta.courses.91', 'meta.courses.92', 'meta.courses.93', 'ratings.content.rating.78', 'ratings.content.rating.79', 'ratings.content.rating.80', 'ratings.content.rating.81', 'ratings.content.rating.82', 'ratings.content.rating.83', 'ratings.content.rating.84', 'ratings.content.rating.85', 'ratings.content.rating.86', 'ratings.content.rating.87', 'ratings.content.rating.88', 'ratings.content.rating.89', 'ratings.content.rating.90', 'ratings.content.rating.91', 'ratings.content.rating.92', 'ratings.content.rating.93', 'ratings.content.rating.

,meta.name,meta.courses.100,meta.courses.101,meta.courses.102,meta.courses.103,meta.courses.94,meta.courses.95,meta.courses.96,meta.courses.97,meta.courses.98,...,ratings.workload.confidence.50,ratings.workload.confidence.51,ratings.workload.samples.48,ratings.workload.samples.49,ratings.workload.samples.50,ratings.workload.samples.51,ratings.workload.bayesian.48,ratings.workload.bayesian.49,ratings.workload.bayesian.50,ratings.workload.bayesian.51
0,"ABDELBAKY, Moustafa Abdelhamid N. M. A.","[{'subject': 'ELEC', 'code': '5130'}]",[],"[{'subject': 'ELEC', 'code': '5160'}]",[],NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"ABDOLLAHI ALIBEIK, Ali",NaN,NaN,"[{'subject': 'COMP', 'code': '5423'}]",[],NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"ABDURRAHIM, Fauzan",[],[],[],[],"[{'subject': 'IEDA', 'code': '2100S'}]",[],[],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"ABEYNAYAKE, H.I. Malinda M.","[{'subject': 'ENGG', 'code': '1300'}]",[],"[{'subject': 'ENGG', 'code': '4950W'}, {'subje...",[],"[{'subject': 'ENGG', 'code': '4930G'}]",[],"[{'subject': 'ENGG', 'code': '1300'}]",[],"[{'subject': 'ENGG', 'code': '4950V'}, {'subje...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"ABEYNAYAKE, Hiddadura Isura Malinda Mendis",[],[],[],[],"[{'subject': 'ENGG', 'code': '4930G'}]",[],"[{'subject': 'ENGG', 'code': '1300'}]",[],"[{'subject': 'UCOP', 'code': '3200'}]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
df['meta.name']

0          ABDELBAKY, Moustafa Abdelhamid N. M. A.
1                           ABDOLLAHI ALIBEIK, Ali
2                               ABDURRAHIM, Fauzan
3                      ABEYNAYAKE, H.I. Malinda M.
4       ABEYNAYAKE, Hiddadura Isura Malinda Mendis
                           ...                    
3223                                     ZU, Fuhao
3224                        ZUERN, Tobias Benedikt
3225                                      ZUO, Cai
3226                          ZWEIG, David Stephen
3227                              ZYCHOWICZ, Piotr
Name: meta.name, Length: 3228, dtype: object

In [44]:
import re
import pandas as pd

# 1. find term info 
term_cols = [c for c in df.columns if c.startswith("ratings.") and ".bayesian." in c]
terms = sorted({int(c.rsplit(".", 1)[-1]) for c in term_cols})

# 2. get the latest term (the largest term number) to use for ranking, and print it out for reference
latest_term = max(terms)
print("latest_term =", latest_term)

criteria = ["content", "teaching", "grading", "workload", "course", "instructor"]

# 3. calculate score based on the formula:
def get_value(row, criterion, field, term):
    col = f"ratings.{criterion}.{field}.{term}"
    return row[col] if col in row.index and pd.notna(row[col]) else 0

def calc_score(row, term):
    # Remove workload to avoid leakage into workload prediction target
    score_wo_workload = (
        get_value(row, "content", "bayesian", term) * 2/3 * 0.4 +
        get_value(row, "teaching", "bayesian", term) * 2/3 * 0.4 +
        get_value(row, "grading", "bayesian", term) * 2/3 * 0.15 +
        get_value(row, "course", "bayesian", term) * 1/3 * 0.25 +
        get_value(row, "instructor", "bayesian", term) * 1/3 * 0.75
    )

    # Old total weight was 1.0; after removing workload (2/3*0.05 = 1/30),
    # remaining weight is 29/30. Normalize back to comparable scale.
    return score_wo_workload / (29/30)

df["score"] = df.apply(lambda row: calc_score(row, latest_term), axis=1)

# 4. rank the instructors by score and calculate percentile
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
df["percentile"] = 1 - df.index / len(df)

# 5. calculate the letter grade based on the percentile 
def letter_grade(percentile):
    for threshold, grade in [
        (0.9, "A+"),
        (0.8, "A"),
        (0.75, "A-"),
        (0.6, "B+"),
        (0.45, "B"),
        (0.35, "B-"),
        (0.3, "C+"),
        (0.25, "C"),
        (0.2, "C-"),
        (0.1, "D"),
        (0.0, "F"),
    ]:
        if percentile >= threshold:
            return grade
    return "F"

# 6. keep the result df 
result = pd.DataFrame({
    "instructor name": df["meta.name"],
    "ranking": df["percentile"].apply(letter_grade),
    "score": df["score"]
})

display(result.head(20))
print(result.shape)

latest_term = 103


,instructor name,ranking,score
0,"TSOI, Yau Chat",A+,0.711526
1,"FONG, Tsz Ho",A+,0.567629
2,"NG, Ka Man",A+,0.528440
3,"BAO, Zhigang",A+,0.525558
4,"ARYA, Sunil",A+,0.514835
5,"WONG, Raymond C W",A+,0.506067
6,"NASON, Stephen William",A+,0.470255
7,"WONG, James K.",A+,0.469099
8,"IP, Ivan Chi Ho",A+,0.445507
9,"CHAN, Gary Shueng Han",A+,0.439623


(3228, 3)


In [45]:
result.to_csv("./ust_reviews/instructor_ranking.csv", index=False, encoding="utf-8-sig")